# Stack

In [5]:
class Stack:
    def __init__(self):
        self.items = []
        
    def push(self, item):
        self.items.append(item)

    def pop(self):
        if not self.is_empty():
            return self.items.pop()
        raise IndexError("pop from empty stack")

    def peek(self):
        if not self.is_empty():
            return self.items[-1]
        raise IndexError("peek from empty stack")

    def is_empty(self):
        return len(self.items) == 0

    def size(self):
        return len(self.items)

 # Queue

In [12]:
from collections import deque
class Queue:
    def __init__(self):
        self.items = deque()

    def enqueue(self, item):    
        self.items.append(item)
    
    def dequeue(self):
        return self.items.popleft()
    
    def is_empty(self):
        return not self.items

    def size(self):
        return len(self.items)
    
    def __str__(self) -> str:
        return str(self.items)
        

# Priority Queue

In [13]:
import heapq
class PriorityQueue:
    def __init__(self):
        self.elements = []
    
    def put(self, item, priority):
        heapq.heappush(self.elements, (priority, item))
    
    def get(self):
        return heapq.heappop(self.elements)[1]
    
    def is_empty(self) -> bool:
        return not self.elements
    
    def size(self) -> int:
        return len(self.elements)
    
    def __str__(self) -> str:
        return str(self.elements)

# Reading a Maze file

In [2]:
def read_maze(file_name):
    try:
        with open(file_name) as f:
            maze = [[char for char in line.strip("\n")] for line in f]
            num_cols_top_row = len(maze[0])
            for row in maze:
                if len(row) != num_cols_top_row:
                    print("The maze is not rectangular")
                    raise SystemExit
            return maze
    except OSError:
        print("There was a problem reading the file")
        raise SystemExit

In [3]:
maze = read_maze("mazes/modest_maze.txt")
for row in maze:
    print(row)

['*', '*', '*', '*', '*', '*', '*', '*', '*', '*']
['*', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', '*']
['*', ' ', '*', ' ', '*', '*', '*', '*', '*', '*']
['*', ' ', '*', ' ', ' ', ' ', ' ', ' ', ' ', '*']
['*', ' ', '*', '*', ' ', '*', ' ', '*', '*', '*']
['*', ' ', '*', ' ', ' ', '*', ' ', '*', ' ', '*']
['*', ' ', '*', ' ', ' ', ' ', ' ', ' ', ' ', '*']
['*', ' ', '*', '*', '*', ' ', '*', '*', '*', '*']
['*', ' ', '*', ' ', ' ', ' ', ' ', ' ', ' ', '*']
['*', '*', '*', '*', '*', '*', '*', '*', '*', '*']


# Depth First Search

In [4]:
offsets = {
    "right": (0, 1),
    "left": (0, -1),
    "up": (-1, 0),
    "down": (1, 0)
}

maze = read_maze("mazes/modest_maze.txt")

def is_legal_pos(maze, pos):
    i, j = pos
    num_rows = len(maze)
    num_cols = len(maze[0])
    return 0 <= i < num_rows and 0 <= j < num_cols and maze[i][j] != "*"

def get_path(predecessors, start, goal) -> list[str]:
    current = goal
    path = []
    while current != start:
        path.append(current)
        current = predecessors[current]
    path.append(start)
    path.reverse()
    return path

In [6]:
def dfs(maze, start, goal):
    stack = Stack()
    predecessors = {start: None}
    stack.push(start)
    while not stack.is_empty():
        current_cell = stack.pop()
        if current_cell == goal:
            return get_path(predecessors=predecessors, start=start, goal=goal)
        for direction in ["up", "right", "down", "left"]:
            row_offset, col_offset = offsets[direction]
            neighbour = (current_cell[0] + row_offset, current_cell[1] + col_offset)
            if is_legal_pos(maze, neighbour) and neighbour not in predecessors:
                stack.push(neighbour)
                predecessors[neighbour] = current_cell

    return None  # No path found

# A* Shortest Distance Algorithm

In [20]:
def heuristic(a, b):
    """
    Calculates the Manhattan distance between two pairs of coordinates.
    """
    x1, y1 = a
    x2, y2 = b
    return abs(x1 - x2) + abs(y1 - y2)

In [23]:
def a_star(maze, start, goal):
    pq = PriorityQueue()
    pq.put(start, 0)
    predecessors = {start: None}
    g_values = {start: 0}
    
    while not pq.is_empty():
        current_cell = pq.get()
        if current_cell == goal:
            return get_path(predecessors, start, goal)
        for direction in ["up", "right", "down", "left"]:
            row_offset, col_offset = offsets[direction]
            neighbour = current_cell[0] + row_offset, current_cell[1] + col_offset
            if is_legal_pos(maze, neighbour) and neighbour not in g_values:
                new_cost = g_values[current_cell] + 1
                g_values[neighbour] = new_cost
                f_value = new_cost + heuristic(goal, neighbour)
                pq.put(neighbour, f_value)
                predecessors[neighbour] = current_cell
    return None  # No path found

In [31]:
# Test Cases for DFS with Maze
maze = [[0] * 3 for row in range(3)]
for row in maze:
    print(row)
start_pos = (0, 0)
goal_pos = (2, 2)
# result = dfs(maze, start_pos, goal_pos)
result = a_star(maze, start_pos, goal_pos)
print(f"ShortestPath: {result}")
assert result == [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)], f"Expected path not found: {result}"

print("\n-------------------------\n")
maze = read_maze("mazes/mini_maze_dfs.txt")
for row in maze:
    print(row)
start_pos = (0, 0)
goal_pos = (2, 2)
# result = dfs(maze, start_pos, goal_pos)
result = a_star(maze, start_pos, goal_pos)
print(f"ShortestPath: {result}")
assert result == [(0,0), (0,1), (1, 1), (2, 1), (2, 2)], f"Expected path not found: {result}"


[0, 0, 0]
[0, 0, 0]
[0, 0, 0]
ShortestPath: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)]

-------------------------

[' ', ' ', ' ']
['*', ' ', '*']
[' ', ' ', ' ']
ShortestPath: [(0, 0), (0, 1), (1, 1), (2, 1), (2, 2)]
